In [2]:
!pip install --upgrade -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 64.1 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 97.5 MB/s eta 0:00:00:00:01


In [3]:
import torch
from transformers import AutoProcessor, Gemma3nForConditionalGeneration

MODEL_ID = "oddadmix/egyptian-code-switching-b4-g2-merged"

def load_model_and_processor(model_id=MODEL_ID, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Loading model {model_id} to device {device}...")
    
    model = Gemma3nForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if device == "cuda" else None,
        device_map="auto" if device == "cuda" else None,
    ).eval()

    if not any(p.device.type == "cuda" for p in model.parameters()) and device == "cuda":
        model.to("cuda")

    processor = AutoProcessor.from_pretrained(model_id)
    return model, processor, device


def transcribe_file(model, processor, audio_path, max_new_tokens=128):
    if not audio_path:
        raise ValueError("audio_path must point to an audio file")

    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": "You are an assistant that transcribes speech accurately."}
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "audio", "url": audio_path},
                {"type": "text", "text": "Please transcribe this audio."}
            ],
        },
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    gen_tokens = generated[0][input_len:]
    text = processor.decode(gen_tokens, skip_special_tokens=True)
    return text

In [4]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("huggingface")
login(secret_value_0)

In [5]:
model, processor, device = load_model_and_processor()

Loading model oddadmix/egyptian-code-switching-b4-g2-merged to device cuda...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1676 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/769 [00:00<?, ?B/s]

In [6]:
print(type(model))
print(type(processor))
print(type(device))
print(model.device)

<class 'transformers.models.gemma3n.modeling_gemma3n.Gemma3nForConditionalGeneration'>
<class 'transformers.models.gemma3n.processing_gemma3n.Gemma3nProcessor'>
<class 'str'>
cuda:0


In [7]:
!ffmpeg -i /kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Sample/ring_tree_topology.wav -ss 00:00:00 -t 20 /kaggle/working/output20.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [8]:
audio_path = '/kaggle/working/output20.wav'
transcription = transcribe_file(model, processor, audio_path, max_new_tokens=256)
print("Transcription:", transcription)

Transcription: تمام واضح صدور ال Terminator هي ال Terminator طيب هلأ أنا بدي أروي على الشكل التالي بالرينج طبع لو جي لاحظ شايفين هلأ هذا الشكل الموجود معي أنا هون هلأ أنا بدي أشييل هذا ال Terminator مع هذا ال Terminator و بدي أربطه لهذا بدي أربط النهائي هذا مع البداية هذا مع


## transcribing long audio by chunking

In [10]:
import torchaudio
import torch
import os

def transcribe_long_audio(
    model, 
    processor, 
    audio_path, 
    chunk_seconds: float = 28.0,      # safe under the 30 s limit
    overlap_seconds: float = 2.5,     # helps avoid cutting words
    max_new_tokens: int = 256,
    temp_dir: str = "/tmp"
):
    """
    Transcribe any length audio with the Gemma3n model by chunking.
    Returns the full concatenated transcription.
    """
    os.makedirs(temp_dir, exist_ok=True)
    
    # 1. Load audio
    waveform, sample_rate = torchaudio.load(audio_path)
    
    # Convert stereo → mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # 2. Resample to 16 kHz (what the audio encoder expects)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
        sample_rate = 16000
    
    # 3. Chunk parameters
    chunk_samples = int(chunk_seconds * sample_rate)
    overlap_samples = int(overlap_seconds * sample_rate)
    step = chunk_samples - overlap_samples
    
    transcriptions = []
    total_samples = waveform.shape[1]
    start = 0
    chunk_idx = 0
    
    print(f" Splitting {total_samples / sample_rate:.1f} s audio into {chunk_seconds}s chunks with {overlap_seconds}s overlap...\n")
    
    while start < total_samples:
        end = min(start + chunk_samples, total_samples)
        chunk_waveform = waveform[:, start:end]
        
        # Skip tiny trailing chunks
        if chunk_waveform.shape[1] < sample_rate * 3:
            break
            
        # Save temporary WAV
        temp_path = os.path.join(temp_dir, f"temp_chunk_{chunk_idx:04d}.wav")
        torchaudio.save(temp_path, chunk_waveform, sample_rate)
        
        # Transcribe this chunk using your existing function
        try:
            text = transcribe_file(model, processor, temp_path, max_new_tokens=max_new_tokens)
            transcriptions.append(text.strip())
            print(f"Chunk {chunk_idx+1:2d} ({(end-start)/sample_rate:.1f}s) → {len(text)} chars")
        except Exception as e:
            print(f"Error on chunk {chunk_idx}: {e}")
        
        start += step
        chunk_idx += 1
    
    # 4. Simple but effective merge
    full_transcription = " ".join(transcriptions)
    
    print(f"\n Finished! Total transcription length: {len(full_transcription)} characters")
    return full_transcription

In [6]:
!ffmpeg -i /kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Sample/ring_tree_topology.wav -ss 00:00:00 -t 120 /kaggle/working/2minutes.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [7]:
import time

start = time.time()

audio_path = '/kaggle/working/2minutes.wav'   # can be hours long

transcription = transcribe_long_audio(
    model, 
    processor, 
    audio_path,
    chunk_seconds=28.0,      # you can try 29.0 if you want to push it
    overlap_seconds=2.5,
    max_new_tokens=256
)

end = time.time()

print(f"\n Execution time: {end - start:.2f} seconds")

print("\n=== FINAL TRANSCRIPTION ===\n")
print(transcription)

 Splitting 120.0 s audio into 28.0s chunks with 2.5s overlap...

Chunk  1 (28.0s) → 316 chars
Chunk  2 (28.0s) → 299 chars
Chunk  3 (28.0s) → 311 chars
Chunk  4 (28.0s) → 908 chars
Chunk  5 (18.0s) → 212 chars

 Finished! Total transcription length: 2050 characters

 Execution time: 108.59 seconds

=== FINAL TRANSCRIPTION ===

تمام واضح شدورر ال Terminator هي ال Terminator طيب هلأ أنا بدي أروي على الشكل التالي بالرينج T&O لاحظ شايفين هلأ هذا الشكل الموجود معي أنا هون هلأ أنا بدي أشييل هذا ال Terminator مع هذا ال Terminator و بدي أربطه لهذا بدي أربط النهائي هذا مع البداية هذا مصار عندي عبارة عن عبارة عن حلقا هذا نسميه Ring T&O ال Ring T&O الرينج topology هذا هو الـ ring topology لاحظ كل حاسب مرتبط بالحاسب اللي بعد كل حاسب مرتبط بالحاسب اللي بعد والحاسب الأخير وين؟ ما عندي حاسب أخير هلقا صاروا كلهم حواسب ورا بعض ما عاد تفرق بين الحاسب الأول والحاسب الأخير صارت عبارة عن رينج حلقة صح؟ طبعتولي إيجابية من أول شي سلبية خلوني أشوف سلبياتها Okay خلونا نشوف سلبياتها الشبكي أكيد هي مو سلبية حاسب 

## Transcribing the entire audio file

In [11]:
import time

start = time.time()

audio_path = '/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Sample/ring_tree_topology.wav'

transcription = transcribe_long_audio(
    model, 
    processor, 
    audio_path,
    chunk_seconds=28.0,     
    overlap_seconds=2.5,
    max_new_tokens=256
)

end = time.time()

print(f"\n Execution time: {end - start:.2f} seconds")

print("\n=== FINAL TRANSCRIPTION ===\n")
print(transcription)

 Splitting 370.0 s audio into 28.0s chunks with 2.5s overlap...

Chunk  1 (28.0s) → 344 chars
Chunk  2 (28.0s) → 301 chars
Chunk  3 (28.0s) → 308 chars
Chunk  4 (28.0s) → 440 chars
Chunk  5 (28.0s) → 328 chars
Chunk  6 (28.0s) → 676 chars
Chunk  7 (28.0s) → 89 chars
Chunk  8 (28.0s) → 445 chars
Chunk  9 (28.0s) → 296 chars
Chunk 10 (28.0s) → 397 chars
Chunk 11 (28.0s) → 979 chars
Chunk 12 (28.0s) → 357 chars
Chunk 13 (28.0s) → 27 chars
Chunk 14 (28.0s) → 321 chars
Chunk 15 (13.0s) → 59 chars

 Finished! Total transcription length: 5381 characters

 Execution time: 290.82 seconds

=== FINAL TRANSCRIPTION ===

تمام واضح شدورر الـ Terminator هي الـ Terminator طيب هلأ أنا بدي أروي على الشكل التالي بالـ Ring Tabela Ogi لاحظ شايفين هل لهذا الشكل الموجود معي أنا هون هلأ أنا بدي أشييل هذا الـ Terminator مع هذا الـ Terminator و بدي أربطه لهذا بدي أربط النهائي هذا مع البداية هذا مصار عندي عبارة عن عبارة عن حلقة هذا نسميه Ring Tabela Ogi الـ Ring Tabela Ogi الرينج topology هذا هو الـ ring topolog